# TechOps Intelligence Platform
## Notebook 04 — Vision Processing Pipeline

**Author:** Sudharshan 
**Phase:** 2 — Document Processing  
**Goal:** Process scanned document images through OCR
          and captioning, then embed into ChromaDB

### What This Notebook Does
1. Load RVL-CDIP scanned document images
2. Caption images using BLIP (what is in this image)
3. Extract text using TrOCR (what text is in this image)
4. Combine caption and extracted text
5. Check if already embedded before processing
6. Store in ChromaDB visuals collection

### Why Vision Matters For TechOps
Scanned runbooks, whiteboard architecture photos,
and handwritten notes are common in real IT ops.
This pipeline makes that content searchable.

In [1]:
import os
import re
import json
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
from tqdm import tqdm
from PIL import Image
import torch

from datasets import load_from_disk
from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration,
    TrOCRProcessor,
    VisionEncoderDecoderModel
)
from sentence_transformers import SentenceTransformer
import chromadb

PROJECT_ROOT = Path("C:/Users/sudha/techops-intelligence")
os.chdir(PROJECT_ROOT)

RAW_IMAGES = PROJECT_ROOT / "data/raw/images"
PROCESSED  = PROJECT_ROOT / "data/processed"
EMBEDDINGS = PROJECT_ROOT / "data/embeddings"

print("Imports complete")
print(f"Project root: {PROJECT_ROOT}")

for name, path in {
    "RAW_IMAGES": RAW_IMAGES,
    "PROCESSED" : PROCESSED,
    "EMBEDDINGS": EMBEDDINGS
}.items():
    status = "found" if path.exists() else "missing"
    print(f"  {name}: {status}")

Imports complete
Project root: C:\Users\sudha\techops-intelligence
  RAW_IMAGES: found
  PROCESSED: found
  EMBEDDINGS: found


## 1. Load Models
BLIP for image captioning
TrOCR for text extraction from images
all-mpnet-base-v2 for embeddings

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Embedding model
print("Loading embedding model...")
embedding_model = SentenceTransformer(
    'sentence-transformers/all-mpnet-base-v2',
    device='cpu'
)
print("Embedding model ready")

# BLIP image captioning
print("Loading BLIP captioning model...")
blip_processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)
blip_model.eval()
print("BLIP model ready")

# TrOCR text extraction
from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    AutoTokenizer
)

print("Loading TrOCR model...")
trocr_processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-printed",
)
trocr_model = VisionEncoderDecoderModel.from_pretrained(
    "microsoft/trocr-base-printed"
)
trocr_model.eval()
print("TrOCR model ready")

Device: cpu
Loading embedding model...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Embedding model ready
Loading BLIP captioning model...
BLIP model ready
Loading TrOCR model...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TrOCR model ready


## 2. Connect ChromaDB
Check existing visuals collection
Skip images already embedded

In [3]:
chroma_path = str(EMBEDDINGS / "chroma_db")
client      = chromadb.PersistentClient(path=chroma_path)

visuals_collection = client.get_or_create_collection(
    name     = "visuals",
    metadata = {"hnsw:space": "cosine"}
)

print(f"ChromaDB connected")
print(f"Visuals collection current count: {visuals_collection.count()}")


def get_existing_ids(collection) -> set:
    """
    Fetch all document IDs already in the collection.
    Used to skip re-embedding documents that already exist.
    """
    if collection.count() == 0:
        return set()

    # Fetch all existing IDs in batches
    existing = collection.get(include=[])
    return set(existing['ids'])


def should_embed(doc_id: str, existing_ids: set) -> bool:
    """
    Return True if this document needs embedding.
    Return False if it already exists in ChromaDB.
    """
    return doc_id not in existing_ids


existing_ids = get_existing_ids(visuals_collection)
print(f"Already embedded: {len(existing_ids)} documents")

ChromaDB connected
Visuals collection current count: 0
Already embedded: 0 documents


## 3. Image Processing Functions
Caption + OCR combined for rich text representation
Each image becomes a text document for retrieval

In [6]:
def generate_caption(image: Image.Image) -> str:
    """
    Generate a natural language caption for an image.
    Describes what the image shows at a high level.
    """
    inputs = blip_processor(
        image,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        output = blip_model.generate(
            **inputs,
            max_new_tokens=50
        )

    caption = blip_processor.decode(
        output[0],
        skip_special_tokens=True
    )
    return caption.strip()


def extract_text_from_image(image: Image.Image) -> str:
    """
    Extract printed text from an image using TrOCR.
    Works best on printed text — handles scanned docs well.
    """
    # TrOCR works on smaller crops — resize if needed
    max_size = 384
    if max(image.size) > max_size * 2:
        image.thumbnail(
            (max_size * 2, max_size * 2),
            Image.LANCZOS
        )

    # Convert to RGB if needed
    if image.mode != 'RGB':
        image = image.convert('RGB')

    pixel_values = trocr_processor(
        image,
        return_tensors="pt"
    ).pixel_values

    with torch.no_grad():
        generated_ids = trocr_model.generate(
            pixel_values,
            max_new_tokens=128
        )

    text = trocr_processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]
    return text.strip()


def build_image_text(
    caption   : str,
    ocr_text  : str,
    label_name: str,
    image_idx : int
) -> str:
    """
    Combine caption and OCR text into a single
    rich text representation for embedding.
    """
    parts = []

    parts.append(f"Document type: {label_name}")

    if caption and len(caption) > 5:
        parts.append(f"Description: {caption}")

    if ocr_text and len(ocr_text) > 5:
        parts.append(f"Text content: {ocr_text}")

    return " | ".join(parts)


def clean_extracted_text(text: str) -> str:
    """Clean OCR output — remove noise characters"""
    text = re.sub(r'[^\w\s\.\,\:\-\/\(\)]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


# Quick test
print("Testing image processing functions...")
print("Functions defined successfully")
print("Will test on actual data in next cell")

Testing image processing functions...
Functions defined successfully
Will test on actual data in next cell


## 4. Load and Process RVL-CDIP Dataset
Check each image ID before processing
Skip if already in ChromaDB
Process and embed new images only

In [7]:
# Load dataset
print("Loading RVL-CDIP dataset...")
ds = load_from_disk(str(RAW_IMAGES / "rvlcdip"))

label_names = ds.features['label'].names
print(f"Dataset loaded: {len(ds)} images")
print(f"Label types: {label_names}")

# Fetch existing IDs upfront — avoids per-image DB calls
existing_ids = get_existing_ids(visuals_collection)
print(f"Already in ChromaDB: {len(existing_ids)} images")

# Processing config
BATCH_SIZE    = 8    # small — vision models are memory heavy
skipped       = 0
processed     = 0
failed        = 0

texts_batch     = []
ids_batch       = []
metadatas_batch = []

print(f"\nStarting processing of {len(ds)} images...")
print("Skipping any images already embedded")

for idx in tqdm(range(len(ds)), desc="Processing images"):
    doc_id     = f"rvlcdip_{idx}"
    label_idx  = ds[idx]['label']
    label_name = label_names[label_idx]

    # Skip if already embedded
    if not should_embed(doc_id, existing_ids):
        skipped += 1
        continue

    try:
        image = ds[idx]['image']

        # Ensure PIL Image
        if not isinstance(image, Image.Image):
            image = Image.fromarray(image)

        # Convert to RGB
        if image.mode != 'RGB':
            image = image.convert('RGB')

        # Generate caption
        caption  = generate_caption(image)

        # Extract text via OCR
        ocr_text = extract_text_from_image(image)
        ocr_text = clean_extracted_text(ocr_text)

        # Build combined text
        combined = build_image_text(
            caption,
            ocr_text,
            label_name,
            idx
        )

        # Skip if no useful content extracted
        if len(combined.strip()) < 20:
            failed += 1
            continue

        texts_batch.append(combined)
        ids_batch.append(doc_id)
        metadatas_batch.append({
            "source"    : "rvlcdip",
            "label"     : label_name,
            "image_idx" : str(idx),
            "doc_type"  : "scanned_document",
            "has_ocr"   : str(len(ocr_text) > 0),
            "has_caption": str(len(caption) > 0)
        })

        # Embed and store when batch is full
        if len(texts_batch) >= BATCH_SIZE:
            embeddings = embedding_model.encode(
                texts_batch,
                show_progress_bar=False
            )
            visuals_collection.upsert(
                documents  = texts_batch,
                embeddings = embeddings.tolist(),
                metadatas  = metadatas_batch,
                ids        = ids_batch
            )
            processed     += len(texts_batch)
            texts_batch     = []
            ids_batch       = []
            metadatas_batch = []

    except Exception as e:
        failed += 1
        if failed <= 5:
            print(f"Error on image {idx}: {e}")

# Store remaining batch
if texts_batch:
    embeddings = embedding_model.encode(
        texts_batch,
        show_progress_bar=False
    )
    visuals_collection.upsert(
        documents  = texts_batch,
        embeddings = embeddings.tolist(),
        metadatas  = metadatas_batch,
        ids        = ids_batch
    )
    processed += len(texts_batch)

print(f"\nProcessing complete")
print(f"  Processed  : {processed}")
print(f"  Skipped    : {skipped} (already in ChromaDB)")
print(f"  Failed     : {failed}")
print(f"  Collection : {visuals_collection.count()} total documents")

Loading RVL-CDIP dataset...
Dataset loaded: 1280 images
Label types: ['advertisement', 'budget', 'email', 'file_folder', 'form', 'handwritten', 'invoice', 'letter', 'memo', 'news_article', 'presentation', 'questionnaire', 'resume', 'scientific_publication', 'scientific_report', 'specification']
Already in ChromaDB: 0 images

Starting processing of 1280 images...
Skipping any images already embedded


Processing images: 100%|██████████| 1280/1280 [1:35:54<00:00,  4.50s/it] 


Processing complete
  Processed  : 1280
  Skipped    : 0 (already in ChromaDB)
  Failed     : 0
  Collection : 1280 total documents


## 5. Process RVL-CDIP OCR Dataset
Pre-extracted OCR text — faster than vision models
Use as ground truth validation for our TrOCR output

In [8]:
# Load OCR dataset
print("Loading RVL-CDIP OCR dataset...")
ds_ocr = load_from_disk(str(RAW_IMAGES / "rvlcdip_ocr"))
print(f"OCR dataset loaded: {len(ds_ocr)} entries")
print(f"Columns: {ds_ocr.column_names}")

# Preview structure
sample = ds_ocr[0]
print(f"\nSample entry keys: {list(sample.keys())}")

# Find the text column
text_col = None
for col in ds_ocr.column_names:
    if 'text' in col.lower() or 'ocr' in col.lower():
        text_col = col
        break

if text_col:
    print(f"Text column found: {text_col}")
    print(f"Sample text: {str(sample[text_col])[:200]}")
else:
    print(f"No text column found")
    print(f"All columns: {ds_ocr.column_names}")
    print(f"Sample values: {sample}")

Loading RVL-CDIP OCR dataset...
OCR dataset loaded: 300 entries
Columns: ['image', 'width', 'height', 'category', 'ocr_words', 'word_boxes', 'ocr_paragraphs', 'paragraph_boxes', 'label']

Sample entry keys: ['image', 'width', 'height', 'category', 'ocr_words', 'word_boxes', 'ocr_paragraphs', 'paragraph_boxes', 'label']
Text column found: ocr_words
Sample text: ['5', 'To', 'Whew', 'Mong', 'O', 'ree', 'guest', 'wee', 'be', 'con', 'ok', '~', 'Fo', 'vote', 'Sahel', 'Dien', 'so', 'a', 'lo!', '~', '=', 'coy', 'much', 'Ton?', 'ABS', 'Ase.', 'SOs', '=', 'owe', 'QO'


In [10]:
# Only run if text column was found
if text_col:
    existing_ids = get_existing_ids(visuals_collection)
    print(f"Already embedded: {len(existing_ids)} docs")

    skipped   = 0
    processed = 0
    failed    = 0

    texts_batch     = []
    ids_batch       = []
    metadatas_batch = []


    for idx in tqdm(range(len(ds_ocr)), desc="OCR dataset"):
        doc_id = f"rvlcdip_ocr_{idx}"

        if not should_embed(doc_id, existing_ids):
            skipped += 1
            continue

        try:
            item     = ds_ocr[idx]
            ocr_text = str(item.get(text_col, ''))

            if len(ocr_text.strip()) < 20:
                failed += 1
                continue

            label_idx = item.get("label", -1)

            category = item.get("category")
            label_name = str(category) if category else f"label_{label_idx}"

            combined = (
                f"Document type: {label_name}. "
                f"Text content: {ocr_text[:500]}"
            )

            texts_batch.append(combined)
            ids_batch.append(doc_id)
            metadatas_batch.append({
                "source"  : "rvlcdip_ocr",
                "label"   : label_name,
                "doc_type": "scanned_document_ocr",
                "idx"     : str(idx)
            })

            if len(texts_batch) >= 32:
                embeddings = embedding_model.encode(
                    texts_batch,
                    show_progress_bar=False
                )
                visuals_collection.upsert(
                    documents  = texts_batch,
                    embeddings = embeddings.tolist(),
                    metadatas  = metadatas_batch,
                    ids        = ids_batch
                )
                processed       += len(texts_batch)
                texts_batch      = []
                ids_batch        = []
                metadatas_batch  = []

        except Exception as e:
            failed += 1

    # Remaining batch
    if texts_batch:
        embeddings = embedding_model.encode(texts_batch)
        visuals_collection.upsert(
            documents  = texts_batch,
            embeddings = embeddings.tolist(),
            metadatas  = metadatas_batch,
            ids        = ids_batch
        )
        processed += len(texts_batch)

    print(f"OCR dataset processing complete")
    print(f"  Processed  : {processed}")
    print(f"  Skipped    : {skipped}")
    print(f"  Failed     : {failed}")
    print(f"  Collection : {visuals_collection.count()} total")

else:
    print("Skipping OCR dataset embed - no text column found")

Already embedded: 1280 docs


OCR dataset: 100%|██████████| 300/300 [02:52<00:00,  1.74it/s]


OCR dataset processing complete
  Processed  : 294
  Skipped    : 0
  Failed     : 6
  Collection : 1574 total


In [11]:
def visual_search(query, collection, model, n=3):
    query_emb = model.encode([query]).tolist()
    results   = collection.query(
        query_embeddings = query_emb,
        n_results        = n,
        include          = ['documents', 'metadatas', 'distances']
    )
    return results


test_queries = [
    "technical document with printed text",
    "handwritten notes diagram",
    "form with structured fields",
    "scientific report with data",
    "memo or letter document"
]

print("Retrieval test on visuals collection\n")

scores = []
for query in test_queries:
    results   = visual_search(
        query, visuals_collection, embedding_model, n=2
    )
    docs      = results['documents'][0]
    distances = results['distances'][0]
    metas     = results['metadatas'][0]
    score     = 1 - distances[0]
    scores.append(score)

    print(f"Query  : {query}")
    print(f"Score  : {score:.3f}")
    print(f"Label  : {metas[0].get('label', 'unknown')}")
    print(f"Source : {metas[0].get('source', 'unknown')}")
    print(f"Text   : {docs[0][:100]}")
    print()

print(f"Score summary")
print(f"  Min : {min(scores):.3f}")
print(f"  Max : {max(scores):.3f}")
print(f"  Avg : {np.mean(scores):.3f}")

Retrieval test on visuals collection

Query  : technical document with printed text
Score  : 0.655
Label  : specification
Source : rvlcdip
Text   : Document type: specification | Description: a document with a handwritten message

Query  : handwritten notes diagram
Score  : 0.670
Label  : handwritten
Source : rvlcdip
Text   : Document type: handwritten | Description: a sheet of paper with a handwritten note

Query  : form with structured fields
Score  : 0.538
Label  : form
Source : rvlcdip
Text   : Document type: form | Description: a form of a form of a form of a form of a form of a form of a for

Query  : scientific report with data
Score  : 0.681
Label  : scientific_report
Source : rvlcdip
Text   : Document type: scientific_report | Description: the document for the application of the application

Query  : memo or letter document
Score  : 0.765
Label  : memo
Source : rvlcdip
Text   : Document type: memo | Description: a typed document from the united states

Score summary
  Min : 0.

In [15]:
print("NOTEBOOK 04 - VISION PIPELINE COMPLETE")
print(f"Visuals collection: {visuals_collection.count()} documents")
print(f"Avg retrieval score: {np.mean(scores):.3f}")

print("\nChromaDB state after notebooks 02 03 04:")

all_collections = client.list_collections()
for col in all_collections:
    c     = client.get_collection(col.name)
    count = c.count()
    print(f"  {col.name:20} : {count:,} documents")

NOTEBOOK 04 - VISION PIPELINE COMPLETE
Visuals collection: 1574 documents
Avg retrieval score: 0.662

ChromaDB state after notebooks 02 03 04:
  logs                 : 0 documents
  playbooks            : 174 documents
  visuals              : 1,574 documents
  incidents            : 20,776 documents
  postmortems          : 292 documents
  knowledge_base       : 10,828 documents
